##### 版權 2024 Google LLC.


In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini 2 - 多工具與多模態即時 API


<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/doggy8088/gemini-api-cookbook/blob/main/gemini-2/plotting_and_mapping.ipynb"><img src="https://ai.google.dev/site-assets/images/docs/colab_logo_32px.png" />在 Google Colab 中執行</a>
  </td>
</table>


在這個筆記本中，你將學習如何使用工具，包括圖表工具、Google 搜尋和程式碼執行於 [Gemini 2.0](https://ai.google.dev/gemini-api/docs/models/gemini-v2) 多模態即時 API。 有關新功能的概述，請參考 [Gemini 2.0 docs](https://ai.google.dev/gemini-api/docs/models/gemini-v2)。

這個筆記本是用 Python 撰寫，並直接使用安全的 Websockets 協議，它*不*使用 GenAI SDK。

如果你不是在尋找程式碼，只想嘗試多媒體串流，請使用 [Live API in Google AI Studio](https://aistudio.google.com/app/live)。


## 設定完成


In [ ]:
%pip install -q 'websockets~=14.0' altair

### 設定你的 API 金鑰

要執行以下Cell，你的 API 金鑰必須儲存在名為 `GOOGLE_API_KEY` 的 Colab Secret 中。如果你尚未擁有 API 金鑰，或不確定如何建立 Colab Secret，請參閱 [驗證](https://github.com/google-gemini/gemini-api-cookbook/blob/main/quickstarts/Authentication.ipynb) 快速入門範例。


In [ ]:
import os
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

多模態即時 API 是隨著 [Gemini 2.0](https://ai.google.dev/gemini-api/docs/models/gemini-v2) 模型推出的新功能。它不會與前一代模型一起使用。

你還需要將客戶端版本設置為 `v1alpha`。


In [ ]:
uri = f"wss://generativelanguage.googleapis.com/ws/google.ai.generativelanguage.v1alpha.GenerativeService.BidiGenerateContent?key={GOOGLE_API_KEY}"
model = "models/gemini-2.0-flash-exp"

### 設定一些輔助工具 

在與 API 互動之前，定義一些你在這個程式碼實驗室中需要的輔助工具。 

在這個筆記本中，你將緩衝串流的 PCM 音訊回應，因此建立一個上下文管理器來包裝 PCM 音訊資料到一個具有相關音訊參數的 wave 音訊檔案中。這樣，你就可以直接在 Colab 中回放音訊。


In [ ]:
import contextlib
import wave

@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
  """Define a wave context manager using the audio parameters supplied."""
  with wave.open(filename, "wb") as wf:
    wf.setnchannels(channels)
    wf.setsampwidth(sample_width)
    wf.setframerate(rate)
    yield wf

使用自訂的記錄器，這樣你就可以輕鬆切換日誌級別，以便檢視來自 API 的即時請求和回應。


In [ ]:
import logging
logger = logging.getLogger("Live")
# Switch to "DEBUG" to see the in-flight requests & responses
logger.setLevel("INFO")

### 定義連接功能

這段程式碼定義了一些功能，將連接到（`quick_connect`）、執行並處理提示（`run`）以及處理特定伺服器回應（`handle_tool_call`，`handle_server_content`）。

這段程式碼使用了 [websockets](https://pypi.org/project/websockets) PyPI 套件，特別是 14.0 中提供的非同步介面，並且不適用於較舊的套件。


In [ ]:
import asyncio
import base64
import json
import time

from websockets.asyncio.client import connect
from IPython import display


async def setup(ws, modality, tools):
  """Perform a setup handshake to configure the conversation."""
  setup = {
      "setup": {
          "model": model,
          "tools": tools,
          "generation_config": {
              "response_modalities": [modality]
          }
      }
  }
  setup_json = json.dumps(setup)
  logger.debug(">>> " + setup_json)
  await ws.send(setup_json)

  setup_response = json.loads(await ws.recv())
  logger.debug("<<< " + json.dumps(setup_response))

async def send(ws, prompt):
  """Send a user content message (only text is supported)."""
  msg = {
    "client_content": {
      "turns": [{"role": "user", "parts": [{"text": prompt}]}],
      "turn_complete": True,
    }
  }
  json_msg = json.dumps(msg)
  logger.debug(">>> " + json_msg)
  await ws.send(json_msg)


def handle_server_content(wf, server_content):
  """Handle any server content messages, e.g. incoming audio or text."""
  audio = False
  model_turn = server_content.pop("modelTurn", None)
  if model_turn:
    text = model_turn["parts"][0].pop("text", None)
    if text:
      print(text, end='')

    inline_data = model_turn['parts'][0].pop('inlineData', None)
    if inline_data:
      print('.', end='')
      b64data = inline_data['data']
      pcm_data = base64.b64decode(b64data)
      wf.writeframes(pcm_data)
      audio = True

  turn_complete = server_content.pop('turnComplete', None)
  return turn_complete, audio


async def handle_tool_call(ws, tool_call, responses):
  """Process an incoming tool call request, returning a response."""
  logger.debug("<<< " + json.dumps(tool_call))
  for fc in tool_call['functionCalls']:

    if fc['name'] in responses:
      # Use a response from `responses` if provided.
      result_entry = responses[fc['name']]
      # If it's a function, actuall call it.
      if callable(result_entry):
        result = result_entry(**fc['args'])
    else:
      # Otherwise it's a stub, just say "OK"
      result = {'string_value': 'ok'}

    msg = {
      'tool_response': {
          'function_responses': [{
              'id': fc['id'],
              'name': fc['name'],
              'response': {'result': result}
          }]
        }
    }
    json_msg = json.dumps(msg)
    logger.debug(">>> " + json_msg)
    await ws.send(json_msg)


@contextlib.asynccontextmanager
async def quick_connect(modality='TEXT', tools=()):
  """Establish a connection and keep it open while the context is active."""
  async with connect(uri, additional_headers={"Content-Type": "application/json"}) as ws:
    await setup(ws, modality, tools)
    yield ws


audio_lock = time.time()

async def run(ws, prompt, responses=()):
  """Send the provided prompt and handle the streamed response."""
  print('>', prompt)
  await send(ws, prompt)

  audio = False
  filename = 'audio.wav'
  with wave_file(filename) as wf:
    async for raw_response in ws:
      response = json.loads(raw_response.decode())
      logger.debug("<<< " + str(response)[:150])

      server_content = response.pop("serverContent", None)
      if server_content:
        turn_complete, a = handle_server_content(wf, server_content)
        audio = audio or a

        if turn_complete:
          print()
          print('<Turn complete>')
          break

      tool_call = response.pop('toolCall', None)
      if tool_call:
        await handle_tool_call(ws, tool_call, responses)

  if audio:
    global audio_lock
    # Sleep before playing audio to make sure we don't play over an existing clip.
    if (delta := audio_lock - time.time()) > 0:
      print('Pausing for audio to complete...')
      await asyncio.sleep(delta + 1.0)  # include a buffer so there's a breather

    display.display(display.Audio(filename, autoplay=True))
    audio_lock = time.time() + (wf.getnframes() / wf.getframerate())

## 使用 API


### 單輪範例

現在，讓我們看一下你定義的所有部分如何在一個簡單的範例中相互配合。你將發送單個提示到 API，並觀察回應。

這個範例使用 `quick_connect` 內容管理器來建立與 API 的連接。只要你在 async with 區塊內，連接將保持活躍，並且可以通過 ws 變數訪問。然後使用 run 函式來發送我們的提示並處理 API 的回應。

發送一個簡單的請求以了解上述程式碼如何運作。通過使用 `quick_connect` 的內容管理器建立連接，當內容有效時，網路插座連接儲存在 `ws` 中，並傳遞給後續的 `run` 呼叫來執行提示。

請注意，你可以將模式從 `AUDIO` 更改為 `TEXT`，並調整提示。


In [ ]:
tools = [
    {'google_search': {}},
    {'code_execution': {}},
]

async def go():
  async with quick_connect(tools=tools, modality="AUDIO") as ws:
    await run(ws, "Please find the last 5 Denis Villeneuve movies and look up their runtimes and the year published.")

logger.setLevel('INFO')
await go()

> Please find the last 5 Denis Villeneuve movies and look up their runtimes and the year published.
....................................................................................................................................................................................................................................
<Turn complete>


### 複雜的多工具範例

現在定義額外的工具。通過定義一個模式（在 `altair_fns` 中），為圖表建立一個工具，執行一個函式（`render_altair`），並使用 `tool_calls` 映射將兩者連接起來。

這裡使用的圖表工具是 [Vega-Altair](https://pypi.org/project/altair/)，這是一個「用於 Python 的宣告式統計視覺化函式庫」。Altair 支援使用 JSON 進行圖表的持久化，你將其作為工具暴露，以便 Gemini 模型可以生成圖表。

之前定義的輔助程式碼將在可以執行時立即運行，但音訊需要一些時間來播放，因此你可能會看到稍後回合的輸出顯示在音訊播放之前。


In [ ]:
import altair as alt
from google.api_core import retry


def apply_altair_theme(altair_json: str, theme: str) -> str:
  chart = alt.Chart.from_json(altair_json)
  with alt.themes.enable(theme):
    themed_altair_json = chart.to_json()
  return themed_altair_json


@retry.Retry()
def render_altair(altair_json: str, theme: str = "default"):
  themed_altair_json = apply_altair_theme(altair_json, theme)
  chart = alt.Chart.from_json(themed_altair_json)
  chart.display()

  return {'string_value': 'ok'}


altair_fns = [
  {
    'name': 'render_altair',
    'description': 'Displays an Altair chart in JSON format.',
    'parameters': {
      'type': 'OBJECT',
      'properties': {
        'altair_json': {
            'type': 'STRING',
            'description': 'JSON STRING representation of the Altair chart to render. Must be a string, not a json object',
        },
        'theme': {
            'type': 'STRING',
            'description': 'Altair theme. Choose from one of "dark", "ggplot2", "default", "opaque".',
        },
      },
    },
  },
]

tool_calls = {
    'render_altair': render_altair,
}

現在將所有內容整合成一個聊天對話。這段程式碼開啟了一個串流會話（使用 `quick_connect`），並且每個 `run` 的呼叫將傳送文字提示，讀取串流回應（若為音訊則進行緩衝），處理任何伺服器回應（例如工具呼叫），並在轉換結束信號被傳送後最終返回。

透過在 `quick_connect` 會話中排定多個 `run` 呼叫，你正在執行一個多輪串流對話。一旦程式碼達到 `quick_connect` 區塊的結尾，會話將被終止。


In [ ]:
tools = [
    {'google_search': {}},
    {'code_execution': {}},
    {'function_declarations': altair_fns},
]

async def go():
  async with quick_connect(tools=tools, modality="AUDIO") as ws:

    # Google Search
    await run(ws, "Please find the last 5 Denis Villeneuve movies and find their runtimes.")
    # Code execution
    await run(ws, "Can you write some code to work out which has the longest and shortest runtimes?")
    # Tool use
    await run(ws, "Now can you plot them in a line chart showing the year on the x-axis and runtime on the y-axis?", responses=tool_calls)
    # Tool use - this step takes user input, so you can ask the model to tweak the chart to your liking.
    # Try changing to dark mode, or lay out the data differently.
    await run(ws, input('Any requests? > '), responses=tool_calls)


logger.setLevel('INFO')
await go()

> Please find the last 5 Denis Villeneuve movies and find their runtimes.
.........................................................................................................................................................................
<Turn complete>


> Can you write some code to work out which has the longest and shortest runtimes?
..................................................
<Turn complete>
Pausing for audio to complete...


> Now can you plot them in a line chart showing the year on the x-axis and runtime on the y-axis?


alt.Chart(...)

..........................................................
<Turn complete>


Any requests? > Very nice! Can you change it to dark mode?
> Very nice! Can you change it to dark mode?


alt.Chart(...)

...........................................................
<Turn complete>


### 地圖範例

在這個範例中，你將使用 [Google Maps Static API](https://developers.google.com/maps/documentation/maps-static) 在對話過程中繪製地圖。你需要 [確保你的 API 金鑰已啟用 Google Maps Static API](https://developers.google.com/maps/documentation/maps-static/get-api-key)。這可以是你用於 Gemini API 的相同 API 金鑰，或者是一個新的，只要啟用了 Static Maps API。

將金鑰添加到 Colab Secrets，或直接在程式碼中添加（`MAPS_API_KEY = 'AIza...'`）。


In [ ]:
from google.colab import userdata
MAPS_API_KEY = userdata.get('MAPS_API_KEY')

以下的單元預設為隱藏，但需要執行。它包含了 `draw_map` 函式的架構，包括一些有關如何使用 Google Maps API 繪製標記的文件。

請注意，模型需要產生一組相當複雜的參數來呼叫 `draw_map`，包括定義地圖的中心點、整數的縮放級別，以及自訂的標記樣式和位置。


In [ ]:
# @title Map tool schema (run this cell)

map_fns = [
  {
    'name': 'draw_map',
    'description': 'Render a Google Maps static map using the specified parameters. No information is returned.',
    'parameters': {
      'type': 'OBJECT',
      'properties': {
        'center': {
            'type': 'STRING',
            'description': 'Location to center the map. It can be a lat,lng pair (e.g. 40.714728,-73.998672), or a string address of a location (e.g. Berkeley,CA).',
        },
        'zoom': {
            'type': 'NUMBER',
            'description': 'Google Maps zoom level. 1 is the world, 20 is zoomed in to building level. Integer only. Level 11 shows about a 15km radius. Level 9 is about 30km radius.'
        },
        'path': {
            "type": "STRING",
            'description': """The path parameter defines a set of one or more locations connected by a path to overlay on the map image. The path parameter takes set of value assignments (path descriptors) of the following format:

path=pathStyles|pathLocation1|pathLocation2|... etc.

Note that both path points are separated from each other using the pipe character (|). Because both style information and point information is delimited via the pipe character, style information must appear first in any path descriptor. Once the Maps Static API server encounters a location in the path descriptor, all other path parameters are assumed to be locations as well.

Path styles
The set of path style descriptors is a series of value assignments separated by the pipe (|) character. This style descriptor defines the visual attributes to use when displaying the path. These style descriptors contain the following key/value assignments:

weight: (optional) specifies the thickness of the path in pixels. If no weight parameter is set, the path will appear in its default thickness (5 pixels).
color: (optional) specifies a color either as a 24-bit (example: color=0xFFFFCC) or 32-bit hexadecimal value (example: color=0xFFFFCCFF), or from the set {black, brown, green, purple, yellow, blue, gray, orange, red, white}.

When a 32-bit hex value is specified, the last two characters specify the 8-bit alpha transparency value. This value varies between 00 (completely transparent) and FF (completely opaque). Note that transparencies are supported in paths, though they are not supported for markers.

fillcolor: (optional) indicates both that the path marks off a polygonal area and specifies the fill color to use as an overlay within that area. The set of locations following need not be a "closed" loop; the Maps Static API server will automatically join the first and last points. Note, however, that any stroke on the exterior of the filled area will not be closed unless you specifically provide the same beginning and end location.
geodesic: (optional) indicates that the requested path should be interpreted as a geodesic line that follows the curvature of the earth. When false, the path is rendered as a straight line in screen space. Defaults to false.
Some example path definitions:

Thin blue line, 50% opacity: path=color:0x0000ff80|weight:1
Solid red line: path=color:0xff0000ff|weight:5
Solid thick white line: path=color:0xffffffff|weight:10
These path styles are optional. If default attributes are desired, you may skip defining the path attributes; in that case, the path descriptor's first "argument" will consist instead of the first declared point (location).

Path points
In order to draw a path, the path parameter must also be passed two or more points. The Maps Static API will then connect the path along those points, in the specified order. Each pathPoint is denoted in the pathDescriptor separated by the | (pipe) character.
""",
        },
        'markers': {
            "type": "ARRAY",
            "items": {
                "type": "STRING"
            },
            # Copied from https://developers.google.com/maps/documentation/maps-static/start#Markers
            'description': """The markers parameter defines a set of one or more markers (map pins) at a set of locations. Each marker defined within a single markers declaration must exhibit the same visual style; if you wish to display markers with different styles, you will need to supply multiple markers parameters with separate style information.

The markers parameter takes set of value assignments (marker descriptors) of the following format:

markers=markerStyles|markerLocation1| markerLocation2|... etc.

The set of markerStyles is declared at the beginning of the markers declaration and consists of zero or more style descriptors separated by the pipe character (|), followed by a set of one or more locations also separated by the pipe character (|).

Because both style information and location information is delimited via the pipe character, style information must appear first in any marker descriptor. Once the Maps Static API server encounters a location in the marker descriptor, all other marker parameters are assumed to be locations as well.

Marker styles
The set of marker style descriptors is a series of value assignments separated by the pipe (|) character. This style descriptor defines the visual attributes to use when displaying the markers within this marker descriptor. These style descriptors contain the following key/value assignments:

size: (optional) specifies the size of marker from the set {tiny, mid, small}. If no size parameter is set, the marker will appear in its default (normal) size.
color: (optional) specifies a 24-bit color (example: color=0xFFFFCC) or a predefined color from the set {black, brown, green, purple, yellow, blue, gray, orange, red, white}.

Note that transparencies (specified using 32-bit hex color values) are not supported in markers, though they are supported for paths.

label: (optional) specifies a single uppercase alphanumeric character from the set {A-Z, 0-9}. (The requirement for uppercase characters is new to this version of the API.) Note that default and mid sized markers are the only markers capable of displaying an alphanumeric-character parameter. tiny and small markers are not capable of displaying an alphanumeric-character.
""",
        }
      },
      "required": [
        "center",
        "zoom",
      ]

    },
  },
]

現在定義 `draw_map` 函式，並將 `google_search` 新增為此對話使用的工具。這將允許模型查詢可能受歡迎的餐廳。


In [ ]:
from urllib.parse import urlencode

import altair as alt
from google.api_core import retry
import requests


def draw_map(center, zoom, path: str = "", markers: list[str] = ()):
  logger.debug(f'MAPS: {center=} {zoom=} {path=} {markers=}')
  q = {
      'key': MAPS_API_KEY,
      'size': '512x512',
      'center': center,
      'zoom': zoom,
  }

  if path:
    q['path'] = path

  qs = list(q.items())

  for marker in markers:
    qs.append(('markers', marker))

  url = f'https://maps.googleapis.com/maps/api/staticmap?{urlencode(qs)}'
  display.display(display.Image(url=url))
  logger.debug(f"Map URL: {url}")

  return {'string_value': f'ok'}


tool_calls = {
    'draw_map': draw_map,
}

tools = [
    {'google_search': {}},
    {'function_declarations': map_fns},
]

最後，定義並執行對話。


In [ ]:
async def go():
  async with quick_connect(tools=tools, modality="TEXT") as ws:

    # Google Search + Tools (Maps)
    await run(ws, "Please look up and mark 3 Sydney restaurants that are currently trending on a map.", responses=tool_calls)
    # Code execution + Tools
    await run(ws, "Now write some code to randomly pick one to eat at tonight and zoom in to that one on the map.", responses=tool_calls)


logger.setLevel('INFO')
await go()

### 地圖與程式碼執行

在這個範例中，你將使用之前定義的 Google Maps 工具，並挑戰模型生成顏色漸層，並用它在地圖上視覺化資料。這個任務需要程式碼執行，因此它也被當作一個工具來包含。

具體來說，你將要求模型繪製澳洲的首都城市，並在圍繞該國的圓形方向上應用兩種顏色之間的漸層，使用 Google Maps 標記。


In [ ]:
from urllib.parse import urlencode

import altair as alt
from google.api_core import retry


tool_calls = {
    'draw_map': draw_map,
}

tools = [
    {'code_execution': {}},
    {'function_declarations': map_fns},
]

async def go():
  async with quick_connect(tools=tools, modality="TEXT") as ws:

    # Code exec and tool use. No search.
    await run(ws, "Plot markers on every capital city in Australia using a gradient between "
                  "Orange and Green. Plan out your steps first, then follow the plan.", responses=tool_calls)

    await run(ws, "Awesome! Can you ensure the gradient is applied smoothly in a circular direction "
                  "around the country?", responses=tool_calls)


logger.setLevel('INFO')
await go()

由於你沒有將生成的影像反饋回模型，因此必須依賴你的回饋來達到完美的輸出。這個範例展示了一個假設性對話的前兩個步驟，實際上你可以持續與模型進行迭代，直到結果符合你的需求。


## 下一步

<a name="next_steps"></a>

本指南展示了在 Websockets 上使用 Multimodal Live API 的更多中階用法。

- 要嘗試多媒體串流，請使用 [Live API 在 Google AI Studio](https://aistudio.google.com/app/live) - 無需程式碼。
- 查看使用 Python SDK 的 [Live API 入門範例](../gemini-2/live_api_starter.ipynb) 或使用 [websockets](../gemini-2/websockets/live_api_starter.ipynb)。
- 這個其他筆記本也有一些酷炫的 [搜尋](../gemini-2/search_tool.ipynb) 範例。
- 嘗試 [在 live API 教學中的工具使用](../gemini-2/live_api_tool_use.ipynb)，了解 Gemini 2.0 的新工具使用能力。

或者僅查看在 [Cookbook](https://github.com/google-gemini/cookbook/blob/main/gemini-2/) 範例中展示的其他 Gemini 2.0 能力。
